# LiveCodeBench prompt experiments (local version)

Each call to `run_experiment(...)` creates `runs/<problem_id>/run_NNN/` containing:

- `prompt_mine.json`, `prompt_lcb.json`: the exact messages sent
- `code_mine_<i>.py`, `code_lcb_<i>.py`: extracted code for each sample
- `diff.json`: AST, token, and copydetect comparison per sample pair
- `scores.json`: pass@1 and per-sample pass/fail for both prompts
- `meta.json`: model, temperature, timestamp, system prompt hash, notes

## One-time setup (terminal, not a notebook cell)

Run these in a terminal, ideally WSL or Linux/macOS because LiveCodeBench's code sandbox uses Unix-only features:

```
git clone https://github.com/LiveCodeBench/LiveCodeBench.git
cd LiveCodeBench
uv venv --python 3.11
source .venv/bin/activate        # Windows: .venv\\Scripts\\activate
uv pip install -e .
uv pip install "anthropic<1.0.0" "datasets<4" zss copydetect openai pandas python-dotenv ipykernel
```

Then select that `.venv` as the notebook kernel. Provide your key either as an environment variable
`OPENAI_KEY`, or in a `.env` file next to this notebook (`OPENAI_KEY=...`, and keep `.env` out of git).
Optionally set `LCB_RUNS_ROOT` to choose where runs are saved (default: `./lcb_runs`).

In [9]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except ImportError:
    pass

RUNS_ROOT = str(Path(os.environ.get("LCB_RUNS_ROOT", "lcb_runs")).resolve())
os.makedirs(RUNS_ROOT, exist_ok=True)
print("Saving runs to:", RUNS_ROOT)

Saving runs to: C:\Users\mandh\Documents\personalProjects\BSS Scholars\lcb_runs


In [10]:
from openai import OpenAI

api_token = os.environ.get("OPENAI_KEY")
assert api_token, "Set OPENAI_KEY as an environment variable or in a .env file next to this notebook."
client = OpenAI(api_key=api_token, base_url="https://openai.rc.asu.edu/v1")

In [11]:
import sys; print(sys.executable)

c:\Users\mandh\Documents\personalProjects\BSS Scholars\.venv\Scripts\python.exe


In [13]:
import pandas as pd
os.chdir(r"C:\Users\mandh\Documents\personalProjects\BSS Scholars\LiveCodeBench")
from lcb_runner.benchmarks.code_generation import load_code_generation_dataset
from lcb_runner.prompts.code_generation import format_prompt_generation
from lcb_runner.lm_styles import LMStyle
from lcb_runner.evaluation.compute_code_generation_metrics import codegen_metrics

problems = load_code_generation_dataset(release_version="release_v5")  # keep this fixed
PROBLEMS_BY_ID = {x.question_id: x for x in problems}

df = pd.DataFrame([{
    "id": x.question_id, "title": x.question_title,
    "platform": x.platform.value, "difficulty": x.difficulty.value,
    "date": x.contest_date.date()} for x in problems])

display(df[(df.platform == "leetcode") & (df.difficulty == "easy")].sort_values("date").head(40))

c:\Users\mandh\Documents\personalProjects\BSS Scholars\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mandh\.cache\huggingface\hub\datasets--livecodebench--code_generation_lite. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\mandh\Documents\personalProjects\BSS Scholars\.venv\Lib\site-p

Loaded 880 problems


,id,title,platform,difficulty,date
16,2777,find-the-distinct-difference-array,leetcode,easy,2023-05-07
9,2727,number-of-senior-citizens,leetcode,easy,2023-05-13
21,2791,find-the-losers-of-the-circular-game,leetcode,easy,2023-05-14
23,2800,minimum-string-length-after-removing-substrings,leetcode,easy,2023-05-21
29,2816,lexicographically-smallest-palindrome,leetcode,easy,2023-05-21
14,2756,buy-two-chocolates,leetcode,easy,2023-05-27
31,2819,remove-trailing-zeros-from-a-string,leetcode,easy,2023-05-28
19,2785,semi-ordered-permutation,leetcode,easy,2023-06-04
33,2825,minimize-string-length,leetcode,easy,2023-06-04
32,2824,check-if-the-number-is-fascinating,leetcode,easy,2023-06-10


## Diff tools (pure Python, no LCB or API dependencies)

In [14]:
import ast, builtins, difflib, io, keyword, re, tokenize

try:
    import zss
except ImportError:
    zss = None

_BUILTINS = set(dir(builtins))


class _Normalizer(ast.NodeTransformer):
    # Renames identifiers by first appearance and strips annotations,
    # so variable naming and type hints do not dominate the distance.
    def __init__(self):
        self.names = {}

    def _alias(self, name):
        if name in _BUILTINS or name in ("self", "cls"):
            return name
        return self.names.setdefault(name, "v%d" % len(self.names))

    def visit_Name(self, node):
        node.id = self._alias(node.id)
        return node

    def visit_arg(self, node):
        node.arg = self._alias(node.arg)
        node.annotation = None
        return node

    def visit_FunctionDef(self, node):
        node.name = self._alias(node.name)
        node.returns = None
        self.generic_visit(node)
        return node

    visit_AsyncFunctionDef = visit_FunctionDef

    def visit_ClassDef(self, node):
        node.name = self._alias(node.name)
        self.generic_visit(node)
        return node


def _strip_docstrings(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.Module, ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)):
            body = node.body
            if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0], "value", None), ast.Constant) \
                    and isinstance(body[0].value.value, str):
                node.body = body[1:] or [ast.Pass()]


def normalize_ast(code):
    tree = ast.parse(code)
    # drop `from typing import ...` (pure type-hint noise)
    tree.body = [n for n in tree.body
                 if not (isinstance(n, ast.ImportFrom) and n.module == "typing")]
    _strip_docstrings(tree)
    tree = _Normalizer().visit(tree)
    ast.fix_missing_locations(tree)
    return tree


def _children(node):
    return [c for c in ast.iter_child_nodes(node) if not isinstance(c, ast.expr_context)]


def _preorder_labels(node):
    out = [type(node).__name__]
    for c in _children(node):
        out.extend(_preorder_labels(c))
    return out


def _to_zss(node):
    z = zss.Node(type(node).__name__)
    for c in _children(node):
        z.addkid(_to_zss(c))
    return z


def ast_diff(code_a, code_b):
    try:
        ta, tb = normalize_ast(code_a), normalize_ast(code_b)
    except SyntaxError as e:
        return {"parse_ok": False, "error": "SyntaxError: %s" % e}

    la, lb = _preorder_labels(ta), _preorder_labels(tb)
    result = {
        "parse_ok": True,
        "nodes_a": len(la),
        "nodes_b": len(lb),
        "identical_after_normalization": ast.dump(ta) == ast.dump(tb),
        "node_sequence_similarity": difflib.SequenceMatcher(None, la, lb, autojunk=False).ratio(),
    }
    if zss is not None:
        d = zss.simple_distance(_to_zss(ta), _to_zss(tb))
        result["tree_edit_distance"] = d
        result["tree_edit_distance_normalized"] = d / max(len(la), len(lb), 1)
    return result


def token_stream(code):
    skip = {tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT,
            tokenize.DEDENT, tokenize.ENDMARKER}
    out = []
    for tok in tokenize.generate_tokens(io.StringIO(code).readline):
        if tok.type in skip:
            continue
        if tok.type == tokenize.NAME and not keyword.iskeyword(tok.string) and tok.string not in _BUILTINS:
            out.append("ID")
        elif tok.type == tokenize.NUMBER:
            out.append("NUM")
        elif tok.type == tokenize.STRING:
            out.append("STR")
        else:
            out.append(tok.string)
    return out


def token_diff(code_a, code_b):
    try:
        a, b = token_stream(code_a), token_stream(code_b)
    except (tokenize.TokenError, IndentationError, SyntaxError) as e:
        return {"tokenize_ok": False, "error": str(e)}
    return {
        "tokenize_ok": True,
        "tokens_a": len(a),
        "tokens_b": len(b),
        "token_similarity": difflib.SequenceMatcher(None, a, b, autojunk=False).ratio(),
    }


def copydetect_compare(path_a, path_b, noise_t=5, guarantee_t=5):
    # Winnowing-based similarity (same idea as MOSS) that runs locally.
    # noise_t / guarantee_t are in characters of the filtered text; the defaults
    # (25) are too long for short LeetCode solutions, so they are lowered here.
    # Returns the fraction of each file covered by matches with the other.
    try:
        import copydetect
        det = copydetect.CopyDetector(noise_t=noise_t, guarantee_t=guarantee_t,
                                      display_t=0, silent=True, autoopen=False)
        det.add_file(str(path_a), type="test")
        det.add_file(str(path_b), type="ref")
        det.run()
        sim_a, sim_b = det.similarity_matrix[0][0]
        overlap = det.get_copied_code_list()
        return {
            "mine_covered_by_lcb": float(sim_a),
            "lcb_covered_by_mine": float(sim_b),
            "overlap_size": int(overlap[0][-1]) if overlap else 0,
            "noise_t": noise_t,
            "guarantee_t": guarantee_t,
        }
    except Exception as e:
        return {"error": "%s: %s" % (type(e).__name__, e)}

## Run management and experiment function

In [15]:
import hashlib, json, re
from datetime import datetime, timezone
from pathlib import Path

MODEL = "gpt-oss-120b"


def get_problem(problem_id):
    p = PROBLEMS_BY_ID.get(str(problem_id))
    assert p is not None, "No problem with id %s" % problem_id
    return p


def call_llm(messages, temperature=0.2):
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
    return r.choices[0].message.content or ""


def extract_code(text):
    blocks = re.findall(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return blocks[-1] if blocks else text   # applied to BOTH prompts


def build_my_messages(p, system_prompt):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": p.question_content + "\n\nStarter code:\n" + p.starter_code},
    ]


def build_lcb_messages(p):
    prompt = format_prompt_generation(p, LMStyle.OpenAIChat)
    return prompt if isinstance(prompt, list) else [{"role": "user", "content": prompt}]


def next_run_dir(problem_id):
    pdir = Path(RUNS_ROOT) / str(problem_id)
    pdir.mkdir(parents=True, exist_ok=True)
    nums = []
    for d in pdir.iterdir():
        m = re.fullmatch(r"run_(\d+)", d.name)
        if d.is_dir() and m:
            nums.append(int(m.group(1)))
    run_dir = pdir / ("run_%03d" % (max(nums, default=0) + 1))
    run_dir.mkdir()
    return run_dir


def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


def write_text(path, text):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def score_codes(p, codes):
    sample = p.get_evaluation_sample()
    metrics, results, _ = codegen_metrics([sample], [codes], k_list=[1],
                                          num_process_evaluate=1, timeout=6)
    per_sample = []
    try:
        for r in results[0]:
            per_sample.append(all(int(x) > 0 for x in r))
    except Exception:
        per_sample = None
    return {"pass@1": float(metrics["pass@1"]), "per_sample_passed": per_sample}


def run_experiment(problem_id, system_prompt, n_samples=1, temperature=0.2,
                   notes="", use_copydetect=True):
    p = get_problem(problem_id)
    run_dir = next_run_dir(problem_id)

    my_messages = build_my_messages(p, system_prompt)
    lcb_messages = build_lcb_messages(p)
    write_json(run_dir / "prompt_mine.json", my_messages)
    write_json(run_dir / "prompt_lcb.json", lcb_messages)
    write_json(run_dir / "meta.json", {
        "problem_id": str(problem_id),
        "problem_title": p.question_title,
        "model": MODEL,
        "temperature": temperature,
        "n_samples": n_samples,
        "system_prompt_sha256": hashlib.sha256(system_prompt.encode("utf-8")).hexdigest(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "notes": notes,
    })

    my_codes = [extract_code(call_llm(my_messages, temperature)) for _ in range(n_samples)]
    lcb_codes = [extract_code(call_llm(lcb_messages, temperature)) for _ in range(n_samples)]
    for i, (a, b) in enumerate(zip(my_codes, lcb_codes)):
        write_text(run_dir / ("code_mine_%d.py" % i), a)
        write_text(run_dir / ("code_lcb_%d.py" % i), b)

    write_json(run_dir / "scores.json", {
        "mine": score_codes(p, my_codes),
        "lcb": score_codes(p, lcb_codes),
    })

    diffs = []
    for i, (a, b) in enumerate(zip(my_codes, lcb_codes)):
        d = {"sample": i, "ast": ast_diff(a, b), "tokens": token_diff(a, b)}
        if use_copydetect:
            d["copydetect"] = copydetect_compare(run_dir / ("code_mine_%d.py" % i),
                                                 run_dir / ("code_lcb_%d.py" % i))
        diffs.append(d)
    write_json(run_dir / "diff.json", diffs)

    return run_dir


def summarize_runs(problem_ids=None):
    # One row per (problem, run, sample) across everything saved so far
    rows = []
    for pdir in sorted(Path(RUNS_ROOT).iterdir()):
        if not pdir.is_dir() or (problem_ids and pdir.name not in map(str, problem_ids)):
            continue
        for rdir in sorted(pdir.glob("run_*")):
            try:
                with open(rdir / "meta.json", encoding="utf-8") as f:
                    meta = json.load(f)
                with open(rdir / "scores.json", encoding="utf-8") as f:
                    scores = json.load(f)
                with open(rdir / "diff.json", encoding="utf-8") as f:
                    diffs = json.load(f)
            except FileNotFoundError:
                continue
            for d in diffs:
                rows.append({
                    "problem": pdir.name, "run": rdir.name, "sample": d["sample"],
                    "notes": meta.get("notes"),
                    "mine_pass@1": scores["mine"]["pass@1"], "lcb_pass@1": scores["lcb"]["pass@1"],
                    "ast_seq_sim": d["ast"].get("node_sequence_similarity"),
                    "ast_ted_norm": d["ast"].get("tree_edit_distance_normalized"),
                    "token_sim": d["tokens"].get("token_similarity"),
                    "cd_mine_in_lcb": d.get("copydetect", {}).get("mine_covered_by_lcb"),
                    "cd_lcb_in_mine": d.get("copydetect", {}).get("lcb_covered_by_mine"),
                })
    return pd.DataFrame(rows)

## Run
Edit `MY_SYSTEM`, give the run a note describing the change, and execute.

In [22]:
MY_SYSTEM = ("Read through the following specification and respond with error-free Python code "
             "that will solve the problem with maximum space and time efficiency. "
             "Stay within the constraints that are provided. Ensure whitespace is utilized and use comments minimally. "
             "Respond with only the Python code and nothing else."
             "Use starter code if applicable.")

PROBLEM_IDS = ["2857","2777","2727","2800"]

for pid in PROBLEM_IDS:
    run_dir = run_experiment(pid, MY_SYSTEM, n_samples=1, temperature=0.2,
                             notes="baseline prompt")
    print("Saved:", run_dir)

Evaluating 1...


100%|██████████| 1/1 [00:00<00:00,  1.53it/s]


Evaluating 1...


100%|██████████| 1/1 [00:00<00:00,  1.63it/s]


Saved: C:\Users\mandh\Documents\personalProjects\BSS Scholars\lcb_runs\2857\run_003
Evaluating 1...


100%|██████████| 1/1 [00:00<00:00,  1.69it/s]


... (7 outputs left out)

100%|██████████| 1/1 [00:00<00:00,  1.65it/s]


Evaluating 1...


100%|██████████| 1/1 [00:00<00:00,  1.67it/s]


Saved: C:\Users\mandh\Documents\personalProjects\BSS Scholars\lcb_runs\2800\run_003


In [19]:
summarize_runs()

,problem,run,sample,notes,mine_pass@1,lcb_pass@1,ast_seq_sim,ast_ted_norm,token_sim,cd_mine_in_lcb,cd_lcb_in_mine
0,2727,run_001,0,baseline prompt,0.0,0.0,1.000000,0.000000,0.990291,1.000000,0.989362
1,2777,run_001,0,baseline prompt,0.0,0.0,0.483092,0.578512,0.576512,0.888393,0.949438
2,2800,run_001,0,baseline prompt,0.0,0.0,0.979866,0.026667,0.967033,0.898649,0.977941
3,2857,run_001,0,baseline prompt,0.0,0.0,0.910714,0.160714,0.929134,0.950980,0.911765
